In [ ]:

# initiate pyspark session
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

spark = SparkSession.builder.appName("PruebaLecturaEscritura").getOrCreate()

# 2. Define test data and an explicit schema
data = [
    (1, "Laptop", 1200.50),
    (2, "Mouse", 25.00),
    (3, "Keyboard", 45.99),
    (4, "Monitor", 300.00),
]

schema = StructType(
    [
        StructField("id", IntegerType(), True),
        StructField("product", StringType(), True),
        StructField("price", DoubleType(), True)
        if "DoubleType" in globals()
        else StructField("price", StringType(), True),
    ]
)

# Create DF in RAM
df_original = spark.createDataFrame(data, ["id", "product", "price"])

# Define the local path where the files will be saved
output_dir = "./data_demo"

# ---------------------------------------------------------
# STEP A: Save the DF currently in RAM to disk, in all 3 formats
# ---------------------------------------------------------
df_original.write.mode("overwrite").option("header", "true").csv(
    f"{output_dir}/csv"
)
df_original.write.mode("overwrite").json(f"{output_dir}/json")
df_original.write.mode("overwrite").parquet(f"{output_dir}/parquet")

print("--- FILES SUCCESSFULLY GENERATED ---")

# ---------------------------------------------------------
# STEP B: Retrieve (read) the data from each format
# ---------------------------------------------------------

# 1. Read CSV
print("\n>>> Reading a CSV:")
df_csv = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{output_dir}/csv")
)
df_csv.printSchema()
df_csv.show()

# 2. Reading a JSON
print("\n>>> Reading a JSON:")
df_json = spark.read.json(f"{output_dir}/json")
df_json.printSchema()
df_json.show()

# 3. Reading a Parquet
print("\n>>> Reading a Parquet:")
df_parquet = spark.read.parquet(f"{ output_dir}/parquet")
df_parquet.printSchema()
df_parquet.show()


--- FILES SUCCESSFULLY GENERATED ---

>>> Reading a CSV:
root
 |-- id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)

+---+--------+------+
| id| product| price|
+---+--------+------+
|  3|Keyboard| 45.99|
|  4| Monitor| 300.0|
|  1|  Laptop|1200.5|
|  2|   Mouse|  25.0|
+---+--------+------+


>>> Reading a JSON:
root
 |-- id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- product: string (nullable = true)

+---+------+--------+
| id| price| product|
+---+------+--------+
|  3| 45.99|Keyboard|
|  4| 300.0| Monitor|
|  1|1200.5|  Laptop|
|  2|  25.0|   Mouse|
+---+------+--------+


>>> Reading a Parquet:
root
 |-- id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)

+---+--------+------+
| id| product| price|
+---+--------+------+
|  3|Keyboard| 45.99|
|  4| Monitor| 300.0|
|  1|  Laptop|1200.5|
|  2|   Mouse|  25.0|
+---+--------+------+

